In [ ]:
!pip install sdv -q
!pip install torch-geometric -q 2>/dev/null; echo "done"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s eta 0:00:00
done


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sdv.single_table import TVAESynthesizer   # TGAN n'est plus dans SDV >= 1.0
import warnings
warnings.filterwarnings("ignore")

print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU uniquement")

GPU : Tesla T4


In [ ]:
!pip install ctgan -q

from ctgan import CTGAN as TGAN_model   # TGAN partage la meme API que CTGAN

print("TGAN pret via ctgan package")
print("Version :", __import__('ctgan').__version__)

TGAN pret via ctgan package
Version : 0.12.1


In [ ]:

save_dir = '/content'
TARGET   = "label_encoded"

df_selected    = pd.read_csv(f'{save_dir}/df_selected.csv')
df_transformed = pd.read_csv(f'{save_dir}/df_transformed.csv')

print("df_selected   :", df_selected.shape)
print("df_transformed:", df_transformed.shape)

df_selected   : (588, 31)
df_transformed: (588, 31)


In [ ]:
def augmenter_tgan(df, target_col, n_synthetic=500, epochs=300):
    discrete_cols = [target_col]
    feature_cols  = [c for c in df.columns if c != target_col]

    tgan = TGAN_model(
        epochs=epochs,
        batch_size=500,
        discriminator_steps=1,
        cuda=True
    )
    tgan.fit(df, discrete_columns=discrete_cols)
    synthetic = tgan.sample(n_synthetic)

    # Aligner les types
    for col in feature_cols:
        synthetic[col] = synthetic[col].astype(df[col].dtype)
    synthetic[target_col] = synthetic[target_col].astype(df[target_col].dtype)

    df_aug = pd.concat([df, synthetic], ignore_index=True)
    print(f"  {len(df)} -> {len(df_aug)} lignes (+{n_synthetic} synthetiques)")
    return df_aug

print("Augmentation df_selected...")
df_selected_tgan = augmenter_tgan(df_selected, TARGET)

print("\nAugmentation df_transformed...")
df_transformed_tgan = augmenter_tgan(df_transformed, TARGET)

Augmentation df_selected...
  588 -> 1088 lignes (+500 synthetiques)

Augmentation df_transformed...
  588 -> 1088 lignes (+500 synthetiques)


In [ ]:
def preparer(df):
    X = StandardScaler().fit_transform(df.drop(columns=[TARGET]).values)
    y = LabelEncoder().fit_transform(df[TARGET].values)
    return X, y

X_sel,   y_sel   = preparer(df_selected)
X_sel_t, y_sel_t = preparer(df_selected_tgan)
X_tra,   y_tra   = preparer(df_transformed)
X_tra_t, y_tra_t = preparer(df_transformed_tgan)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultats_tgan = {}

def evaluer(nom, model, X_sel, y_sel, X_sel_aug, y_sel_aug,
                        X_tra, y_tra, X_tra_aug, y_tra_aug):
    def scores(X, y):
        preds = cross_val_predict(model, X, y, cv=cv)
        return {
            "Acc": round(accuracy_score(y, preds), 4),
            "F1" : round(f1_score(y, preds, average='weighted'), 4),
        }
    s1 = scores(X_sel,     y_sel)
    s2 = scores(X_sel_aug, y_sel_aug)
    s3 = scores(X_tra,     y_tra)
    s4 = scores(X_tra_aug, y_tra_aug)

    print(f"\n{nom}")
    print(f"{'':22} {'sel_avant':>12} {'sel_apres':>12} {'tra_avant':>12} {'tra_apres':>12}")
    print(f"  {'Accuracy':<20} {s1['Acc']:>12} {s2['Acc']:>12} {s3['Acc']:>12} {s4['Acc']:>12}")
    print(f"  {'F1-score':<20} {s1['F1']:>12} {s2['F1']:>12} {s3['F1']:>12} {s4['F1']:>12}")

    return {"sel_avant": s1, "sel_apres": s2, "tra_avant": s3, "tra_apres": s4}

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x.unsqueeze(1))

def evaluer_cnn_h(X_sel, y_sel, X_sel_aug, y_sel_aug,
                  X_tra, y_tra, X_tra_aug, y_tra_aug, epochs=30):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def run(X, y):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        all_preds, all_true = [], []
        for tr, val in skf.split(X, y):
            X_tr  = torch.FloatTensor(X[tr]).to(device)
            y_tr  = torch.LongTensor(y[tr]).to(device)
            X_val = torch.FloatTensor(X[val]).to(device)
            m     = CNN1D(X.shape[1], len(np.unique(y))).to(device)
            opt   = torch.optim.Adam(m.parameters(), lr=1e-3)
            loss  = nn.CrossEntropyLoss()
            loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
            m.train()
            for _ in range(epochs):
                for xb, yb in loader:
                    opt.zero_grad(); loss(m(xb), yb).backward(); opt.step()
            m.eval()
            with torch.no_grad():
                preds = m(X_val).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds); all_true.extend(y[val])
        return {
            "Acc": round(accuracy_score(all_true, all_preds), 4),
            "F1" : round(f1_score(all_true, all_preds, average='weighted'), 4),
        }

    s1,s2,s3,s4 = run(X_sel,y_sel), run(X_sel_aug,y_sel_aug), run(X_tra,y_tra), run(X_tra_aug,y_tra_aug)
    print(f"\nCNN 1D")
    print(f"{'':22} {'sel_avant':>12} {'sel_apres':>12} {'tra_avant':>12} {'tra_apres':>12}")
    print(f"  {'Accuracy':<20} {s1['Acc']:>12} {s2['Acc']:>12} {s3['Acc']:>12} {s4['Acc']:>12}")
    print(f"  {'F1-score':<20} {s1['F1']:>12} {s2['F1']:>12} {s3['F1']:>12} {s4['F1']:>12}")
    return {"sel_avant": s1, "sel_apres": s2, "tra_avant": s3, "tra_apres": s4}

In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
resultats_tgan["KNN"] = evaluer("KNN", model,
    X_sel, y_sel, X_sel_t, y_sel_t,
    X_tra, y_tra, X_tra_t, y_tra_t)


KNN
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8316       0.7077       0.8435       0.6774
  F1-score                   0.8315       0.7074       0.8434       0.6763


In [ ]:
model = SVC(kernel='rbf', random_state=42)
resultats_tgan["SVM"] = evaluer("SVM", model,
    X_sel, y_sel, X_sel_t, y_sel_t,
    X_tra, y_tra, X_tra_t, y_tra_t)


SVM
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8452       0.7197        0.852       0.7086
  F1-score                    0.845       0.7192       0.8522       0.7088


In [ ]:
model = DecisionTreeClassifier(random_state=42)
resultats_tgan["Decision Tree"] = evaluer("Decision Tree", model,
    X_sel, y_sel, X_sel_t, y_sel_t,
    X_tra, y_tra, X_tra_t, y_tra_t)


Decision Tree
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8146        0.693       0.8163       0.6939
  F1-score                   0.8145       0.6931       0.8162       0.6939


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
resultats_tgan["Logistic Regression"] = evaluer("Logistic Regression", model,
    X_sel, y_sel, X_sel_t, y_sel_t,
    X_tra, y_tra, X_tra_t, y_tra_t)


Logistic Regression
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8418       0.6829       0.8622       0.6921
  F1-score                   0.8419       0.6826       0.8622       0.6922


In [ ]:
model = GaussianNB()
resultats_tgan["Naive Bayes"] = evaluer("Naive Bayes", model,
    X_sel, y_sel, X_sel_t, y_sel_t,
    X_tra, y_tra, X_tra_t, y_tra_t)


Naive Bayes
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.7738       0.6452       0.7313       0.6268
  F1-score                   0.7733       0.6446       0.7313       0.6091


In [ ]:
resultats_tgan["CNN 1D"] = evaluer_cnn_h(
    X_sel,   y_sel,   X_sel_t, y_sel_t,
    X_tra,   y_tra,   X_tra_t, y_tra_t)


CNN 1D
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.7415       0.6517       0.7092       0.6259
  F1-score                   0.7407       0.6457       0.7074       0.6243


In [ ]:
print(f"\n{'TABLEAU RECAPITULATIF TGAN — F1-score':^80}")
print(f"{'='*80}")
print(f"{'Modele':<22} {'sel_avant':>10} {'sel_apres':>10} {'delta_sel':>10} {'tra_avant':>10} {'tra_apres':>10} {'delta_tra':>10}")
print(f"{'-'*80}")
for nom, s in resultats_tgan.items():
    d_sel = round(s["sel_apres"]["F1"] - s["sel_avant"]["F1"], 4)
    d_tra = round(s["tra_apres"]["F1"] - s["tra_avant"]["F1"], 4)
    print(f"  {nom:<20} {s['sel_avant']['F1']:>10} {s['sel_apres']['F1']:>10} {d_sel:>10} {s['tra_avant']['F1']:>10} {s['tra_apres']['F1']:>10} {d_tra:>10}")
print(f"{'='*80}")

df_tgan_result = pd.DataFrame([
    {"Modele": nom,
     "sel_avant": s["sel_avant"]["F1"], "sel_apres": s["sel_apres"]["F1"],
     "delta_sel": round(s["sel_apres"]["F1"] - s["sel_avant"]["F1"], 4),
     "tra_avant": s["tra_avant"]["F1"], "tra_apres": s["tra_apres"]["F1"],
     "delta_tra": round(s["tra_apres"]["F1"] - s["tra_avant"]["F1"], 4)}
    for nom, s in resultats_tgan.items()
])
df_tgan_result.to_csv(f'{save_dir}/resultats_tgan.csv', index=False)
print("\nResultats sauvegardes sur Drive")


                     TABLEAU RECAPITULATIF TGAN — F1-score                      
Modele                  sel_avant  sel_apres  delta_sel  tra_avant  tra_apres  delta_tra
--------------------------------------------------------------------------------
  KNN                      0.8315     0.7074    -0.1241     0.8434     0.6763    -0.1671
  SVM                       0.845     0.7192    -0.1258     0.8522     0.7088    -0.1434
  Decision Tree            0.8145     0.6931    -0.1214     0.8162     0.6939    -0.1223
  Logistic Regression      0.8419     0.6826    -0.1593     0.8622     0.6922      -0.17
  Naive Bayes              0.7733     0.6446    -0.1287     0.7313     0.6091    -0.1222
  CNN 1D                   0.7407     0.6457     -0.095     0.7074     0.6243    -0.0831

Resultats sauvegardes sur Drive
